In [1]:
import pandas as pd
import os

# ==================== 1. 读取原始数据（带类型优化） ====================
print("="*60)
print("阶段2：数据清洗")
print("="*60)

col_names = ["user_id", "item_id", "category_id", "behavior", "timestamp"]

df = pd.read_csv(
    "raw_data/UserBehavior.csv",   # 确保路径正确
    names=col_names,
    dtype={
        "user_id": "int32",
        "item_id": "int32",
        "category_id": "int32",
        "behavior": "category"
    }
)

original_count = len(df)
print(f"✅ 原始数据加载成功，总行数：{original_count:,}")

# ==================== 2. 清洗前快速探查 ====================
print("\n--- 清洗前探查 ---")
print(f"重复行数：{df.duplicated().sum():,}")
print(f"缺失值：\n{df.isnull().sum()}")
print(f"时间戳范围：{df['timestamp'].min()} ~ {df['timestamp'].max()}")

# ==================== 3. 执行清洗 ====================
print("\n--- 开始清洗 ---")

# 3.1 删除重复行
dup_count = df.duplicated().sum()
if dup_count > 0:
    df = df.drop_duplicates()
    print(f"✓ 删除重复行：{dup_count:,} 行")
else:
    print("✓ 无重复行")

# 3.2 时间戳转换
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
print("✓ 时间戳转换完成")

# 3.3 检查并过滤异常年份（只保留2017年数据）
df['year'] = df['datetime'].dt.year
invalid_year = df['year'] != 2017
invalid_year_count = invalid_year.sum()
if invalid_year_count > 0:
    df = df[~invalid_year].copy()
    print(f"✓ 删除非2017年数据：{invalid_year_count:,} 行")
df.drop('year', axis=1, inplace=True)

# 3.4 提取日期、小时、星期
df['date'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour
df['weekday'] = df['datetime'].dt.dayofweek   # 0=周一, 6=周日
print("✓ 已提取 date, hour, weekday")

# 3.5 筛选指定日期范围（2017-11-25 至 2017-12-03）
start_date = pd.to_datetime('2017-11-25').date()
end_date = pd.to_datetime('2017-12-03').date()
before_filter = len(df)
df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
after_filter = len(df)
filtered_out = before_filter - after_filter
print(f"✓ 按日期范围筛选：删除 {filtered_out:,} 行，保留 {after_filter:,} 行")

# 3.6 过滤非法行为类型（确保只有 pv, buy, cart, fav）
valid_behaviors = ['pv', 'buy', 'cart', 'fav']
invalid_behavior = ~df['behavior'].isin(valid_behaviors)
invalid_behavior_count = invalid_behavior.sum()
if invalid_behavior_count > 0:
    df = df[~invalid_behavior]
    print(f"✓ 删除无效行为类型：{invalid_behavior_count:,} 行")

# ==================== 4. 保存清洗后数据 ====================
os.makedirs("processed_data", exist_ok=True)
output_path = "processed_data/UserBehavior_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"\n✅ 清洗后数据已保存至：{output_path}")
print(f"最终数据量：{len(df):,} 行，{df.shape[1]} 列")
print(f"最终时间范围：{df['datetime'].min()} 至 {df['datetime'].max()}")

# ==================== 5. 自动生成清洗日志（简历加分项） ====================
log_content = f"""# 数据清洗日志

## 原始数据
- 来源：raw_data/UserBehavior.csv
- 原始总行数：{original_count:,}

## 清洗步骤及处理量
| 步骤 | 删除行数 | 剩余行数 |
|------|----------|----------|
| 删除重复行 | {dup_count:,} | {original_count - dup_count:,} |
| 删除非2017年数据 | {invalid_year_count:,} | {original_count - dup_count - invalid_year_count:,} |
| 筛选日期范围 (2017-11-25 ~ 2017-12-03) | {filtered_out:,} | {after_filter:,} |
| 删除无效行为类型 | {invalid_behavior_count:,} | {len(df):,} |

## 最终数据
- 输出路径：{output_path}
- 最终行数：{len(df):,}
- 字段列表：{list(df.columns)}
- 时间范围：{df['datetime'].min()} 至 {df['datetime'].max()}

## 备注
- 时间戳单位：秒
- 新增字段：datetime, date, hour, weekday
- 行为类型仅保留 pv, buy, cart, fav
"""

os.makedirs("docs", exist_ok=True)
with open("docs/data_cleaning_log.md", "w", encoding="utf-8") as f:
    f.write(log_content)
print("✅ 清洗日志已保存至 docs/data_cleaning_log.md")

print("\n🎉 阶段2（数据清洗）全部完成！")

阶段2：数据清洗
✅ 原始数据加载成功，总行数：100,150,807

--- 清洗前探查 ---
重复行数：49
缺失值：
user_id        0
item_id        0
category_id    0
behavior       0
timestamp      0
dtype: int64
时间戳范围：-2134949234 ~ 2122867355

--- 开始清洗 ---
✓ 删除重复行：49 行
✓ 时间戳转换完成
✓ 删除非2017年数据：1,876 行
✓ 已提取 date, hour, weekday
✓ 按日期范围筛选：删除 1,234,398 行，保留 98,914,484 行

✅ 清洗后数据已保存至：processed_data/UserBehavior_cleaned.csv
最终数据量：98,914,484 行，9 列
最终时间范围：2017-11-25 00:00:00 至 2017-12-03 23:52:41
✅ 清洗日志已保存至 docs/data_cleaning_log.md

🎉 阶段2（数据清洗）全部完成！
